In [7]:
import numpy as np
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import psutil
import shutil
from pathlib import Path
from datetime import datetime, timedelta
import random


In [9]:
RAIZ = Path('..').resolve()
RUTA_DATOS = RAIZ / 'datos'
RUTA_VISUALI = RAIZ / 'visuali'
for tamano in ['1M', '10M', '100M']:
    (RUTA_DATOS / f'parquet_{tamano}').mkdir(parents=True, exist_ok=True)

ALGORITMOS = ['montecarlo', 'leibniz', 'machin', 'newton', 'simpson', 'trapecio', 'riemann']
UBICACIONES = ['santiago', 'valparaiso', 'concepcion', 'antofagasta', 'temuco']
N_SERVIDORES = 20

TAMANOS = {
    '1M': 1_000_000,
    '10M': 10_000_000,
    '100M': 100_000_000,
}

N_PARTICIONES = [4, 16, 64]
N_REPETICIONES = 3


In [10]:
def generar_servidores(n=20, seed=1):
    rng = np.random.default_rng(seed)
    servidores = pd.DataFrame({
        'id_servidor': range(1, n + 1),
        'nombre': [f'nodo-{i:03d}' for i in range(1, n + 1)],
        'n_cores': rng.choice([4, 8, 16, 32], size=n),
        'ram_gb': rng.choice([16, 32, 64, 128], size=n),
        'ubicacion': rng.choice(UBICACIONES, size=n),
    })
    return servidores

# Generar y guardar
servidores = generar_servidores()
ruta_servidores = RUTA_DATOS / 'servidores.parquet'
servidores.to_parquet(ruta_servidores, index=False)
print(f"Servidores guardados: {ruta_servidores}")
print(servidores.head(10))

Servidores guardados: /Users/melvin/U/INFB6074-infra/Actividad3/datos/servidores.parquet
   id_servidor    nombre  n_cores  ram_gb    ubicacion
0            1  nodo-001        8     128     santiago
1            2  nodo-002       16     128  antofagasta
2            3  nodo-003       32     128     santiago
3            4  nodo-004       32      64   valparaiso
4            5  nodo-005        4     128   concepcion
5            6  nodo-006        4      32   concepcion
6            7  nodo-007       32      32     santiago
7            8  nodo-008       32     128       temuco
8            9  nodo-009        4      16  antofagasta
9           10  nodo-010        8      32       temuco


In [11]:
def generar_ejecuciones_particionado(n_filas, ruta_destino, seed=1):
    """Genera n_filas de ejecuciones y las guarda particionadas por algoritmo."""
    rng = np.random.default_rng(seed)
    
    # Limpiar carpeta antes de generar (por si ya tiene archivos viejos)
    if ruta_destino.exists():
        shutil.rmtree(ruta_destino)
    ruta_destino.mkdir(parents=True, exist_ok=True)
    
    # Fecha base para timestamps sintéticos
    fecha_base = datetime(2024, 1, 1)
    
    # Generar todo el DataFrame de una vez (más rápido que por chunks para estos tamaños)
    df = pd.DataFrame({
        'id_ejecucion': np.arange(1, n_filas + 1),
        'fecha': [fecha_base + timedelta(seconds=int(s)) 
                  for s in rng.integers(0, 365 * 24 * 3600, size=n_filas)],
        'algoritmo': rng.choice(ALGORITMOS, size=n_filas),
        'id_servidor': rng.integers(1, N_SERVIDORES + 1, size=n_filas),
        'iteraciones': rng.integers(1000, 100_000, size=n_filas),
        'error': rng.exponential(0.005, size=n_filas),  # mayoría errores chicos
        'tiempo_ms': rng.exponential(50, size=n_filas) + 1,  # mayoría tiempos cortos
    })
    
    # Guardar particionado por algoritmo
    df.to_parquet(ruta_destino, partition_cols=['algoritmo'], index=False)
    
    return n_filas

# Probar con el dataset chico primero
print("Generando dataset 1M...")
inicio = time.perf_counter()
generar_ejecuciones_particionado(TAMANOS['1M'], RUTA_DATOS / 'parquet_1M')
print(f"Tiempo: {time.perf_counter() - inicio:.1f}s")

# Verificar que se crearon las carpetas particionadas
import os
for item in sorted(os.listdir(RUTA_DATOS / 'parquet_1M')):
    print(f"  {item}")

Generando dataset 1M...
Tiempo: 0.5s
  algoritmo=leibniz
  algoritmo=machin
  algoritmo=montecarlo
  algoritmo=newton
  algoritmo=riemann
  algoritmo=simpson
  algoritmo=trapecio


In [12]:
# Generar 10M
print("Generando dataset 10M...")
inicio = time.perf_counter()
generar_ejecuciones_particionado(TAMANOS['10M'], RUTA_DATOS / 'parquet_10M')
print(f"Tiempo 10M: {time.perf_counter() - inicio:.1f}s")

# Generar 100M
print("\nGenerando dataset 100M...")
inicio = time.perf_counter()
generar_ejecuciones_particionado(TAMANOS['100M'], RUTA_DATOS / 'parquet_100M')
print(f"Tiempo 100M: {time.perf_counter() - inicio:.1f}s")

Generando dataset 10M...
Tiempo 10M: 5.3s

Generando dataset 100M...
Tiempo 100M: 138.9s


In [13]:
def pipeline_pandas(ruta_ejecuciones, ruta_servidores):
    """Pipeline monolítico con Pandas: lee TODO en memoria."""
    # 1. Lectura
    ejecuciones = pd.read_parquet(ruta_ejecuciones)
    servidores = pd.read_parquet(ruta_servidores)
    
    # 2. Transformación: filtro + columna derivada
    ejecuciones = ejecuciones[ejecuciones['error'] < 0.01].copy()
    ejecuciones['eficiencia'] = ejecuciones['iteraciones'] / ejecuciones['tiempo_ms']
    
    # 3. Combinación: merge
    df = ejecuciones.merge(servidores, on='id_servidor')
    
    # 4. Groupby multi-key + agregaciones múltiples
    resultado = df.groupby(['algoritmo', 'ubicacion']).agg(
        n_ejecuciones=('id_ejecucion', 'count'),
        error_promedio=('error', 'mean'),
        eficiencia_promedio=('eficiencia', 'mean'),
        iteraciones_totales=('iteraciones', 'sum'),
        tiempo_p95=('tiempo_ms', lambda x: x.quantile(0.95)),
    ).reset_index()
    
    return resultado


def pipeline_dask(ruta_ejecuciones, ruta_servidores, n_particiones):
    """Pipeline distribuido con Dask: lee particionado, procesa en paralelo."""
    # 1. Lectura (Dask con N particiones)
    ejecuciones = dd.read_parquet(ruta_ejecuciones).repartition(npartitions=n_particiones)
    # Servidores es chico, lo cargamos como Pandas (Dask hace broadcast automático)
    servidores = pd.read_parquet(ruta_servidores)
    
    # 2. Transformación
    ejecuciones = ejecuciones[ejecuciones['error'] < 0.01]
    ejecuciones['eficiencia'] = ejecuciones['iteraciones'] / ejecuciones['tiempo_ms']
    
    # 3. Merge con la tabla chica (broadcast join)
    df = ejecuciones.merge(servidores, on='id_servidor')
    
    # 4. Groupby + agregaciones
    # NOTA: tiempo_p95 con quantile es complejo en Dask, usamos mean en su lugar para Dask
    resultado = df.groupby(['algoritmo', 'ubicacion']).agg(
        n_ejecuciones=('id_ejecucion', 'count'),
        error_promedio=('error', 'mean'),
        eficiencia_promedio=('eficiencia', 'mean'),
        iteraciones_totales=('iteraciones', 'sum'),
        tiempo_promedio=('tiempo_ms', 'mean'),  # mean en lugar de p95 para Dask
    ).reset_index()
    
    # 5. Computar (Dask es lazy, esto fuerza la ejecución)
    return resultado.compute()

In [14]:
print("=== Pandas con 1M ===")
inicio = time.perf_counter()
res_pd = pipeline_pandas(RUTA_DATOS / 'parquet_1M', RUTA_DATOS / 'servidores.parquet')
t_pd = time.perf_counter() - inicio
print(f"Tiempo: {t_pd:.2f}s")
print(f"Filas resultado: {len(res_pd)}")
print(res_pd.head())

print("\n=== Dask con 1M, 4 particiones ===")
inicio = time.perf_counter()
res_dk = pipeline_dask(RUTA_DATOS / 'parquet_1M', RUTA_DATOS / 'servidores.parquet', n_particiones=4)
t_dk = time.perf_counter() - inicio
print(f"Tiempo: {t_dk:.2f}s")
print(f"Filas resultado: {len(res_dk)}")
print(res_dk.head())

=== Pandas con 1M ===
Tiempo: 0.28s
Filas resultado: 35
  algoritmo    ubicacion  n_ejecuciones  error_promedio  eficiencia_promedio  \
0   leibniz  antofagasta          24745        0.003435          3472.103598   
1   leibniz   concepcion          18594        0.003447          3439.653553   
2   leibniz     santiago          30578        0.003425          3525.508032   
3   leibniz       temuco          24793        0.003409          3449.023957   
4   leibniz   valparaiso          24725        0.003415          3452.243479   

   iteraciones_totales  tiempo_p95  
0           1238332914  150.714343  
1            937572100  151.161364  
2           1549929461  151.970981  
3           1250766601  149.002033  
4           1247647857  150.649773  

=== Dask con 1M, 4 particiones ===
Tiempo: 1.01s
Filas resultado: 35
  algoritmo    ubicacion  n_ejecuciones  error_promedio  eficiencia_promedio  \
0   leibniz       temuco          24793        0.003409          3449.023957   
1   leibniz